# pendulum

## Example 1: parse ISO 8601 with the compiled parser

In [ ]:
# Example 1: pendulum.parse goes through the Rust extension (pendulum._pendulum) and keeps the offset
import pendulum
import pendulum.parsing
from pendulum import _pendulum

dt = pendulum.parse("2026-11-01T12:30:45.123456+02:00")
assert (dt.year, dt.month, dt.day, dt.hour, dt.minute, dt.microsecond) == (2026, 11, 1, 12, 30, 123456)
assert dt.utcoffset().total_seconds() == 7200
assert pendulum.parsing.parse_iso8601 is _pendulum.parse_iso8601  # the Rust one, not parsing.iso8601's fallback
print("Example 1:", dt.isoformat(), "| parser from", _pendulum.__name__)

## Example 2: convert across a daylight-saving change

In [ ]:
# Example 2: adding hours across the US DST end keeps wall-clock time correct in America/New_York
import pendulum

before = pendulum.datetime(2026, 11, 1, 0, 30, tz="America/New_York")
after = before.add(hours=3)
assert before.utcoffset().total_seconds() == -4 * 3600 and after.utcoffset().total_seconds() == -5 * 3600
assert (after.hour, after.minute) == (2, 30), after
assert after.in_timezone("UTC").hour == 7 and after.in_timezone("Asia/Tokyo").hour == 16
print("Example 2:", before.to_iso8601_string(), "+3h ->", after.to_iso8601_string())

## Example 3: durations and intervals

In [ ]:
# Example 3: a duration has calendar-aware parts; an interval between two datetimes counts business days
import pendulum

d = pendulum.duration(days=5, hours=3, minutes=30)
assert d.in_hours() == 123 and d.days == 5 and d.hours == 3 and d.minutes == 30
start, end = pendulum.datetime(2026, 1, 1), pendulum.datetime(2026, 3, 1)
interval = end - start
assert interval.in_days() == 59 and interval.months == 2
weekdays = [day for day in interval.range("days") if day.day_of_week not in (pendulum.SATURDAY, pendulum.SUNDAY)]
assert len(weekdays) == 42, len(weekdays)
print("Example 3:", d.in_words(), "|", interval.in_days(), "days,", len(weekdays), "weekdays")

## Example 4: formatting and parsing a custom format

In [ ]:
# Example 4: format() tokens and from_format() are inverses
import pendulum

dt = pendulum.datetime(2026, 6, 15, 14, 30, 45, tz="Europe/Paris")
text = dt.format("dddd, MMMM Do YYYY HH:mm:ss Z")
assert text == "Monday, June 15th 2026 14:30:45 +02:00", text
back = pendulum.from_format("15/06/2026 14:30", "DD/MM/YYYY HH:mm", tz="Europe/Paris")
assert back == dt.replace(second=0)
print("Example 4:", text, "| from_format:", back.to_datetime_string())

## Example 5: period boundaries and comparisons

In [ ]:
# Example 5: start_of / end_of, next weekday, and is_* helpers
import pendulum

base = pendulum.datetime(2026, 2, 14, 10, 30, tz="UTC")
assert base.start_of("month") == pendulum.datetime(2026, 2, 1, tz="UTC")
assert base.end_of("month").day == 28 and not base.is_leap_year()
assert base.next(pendulum.MONDAY).to_date_string() == "2026-02-16"
assert base.quarter == 1 and base.week_of_year == 7
print("Example 5:", base.start_of("week").to_date_string(), "..", base.end_of("week").to_date_string(), "quarter", base.quarter)